In [1]:
!pip install -q langchain-text-splitters chromadb sentence-transformers groq google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

In [2]:
import os
import getpass

def get_key(name):
    key = os.environ.get(name, "")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get(name) or ""
        except Exception:
            key = ""
    if not key:
        key = getpass.getpass(f"Enter {name} (press Enter to skip): ").strip()
    return key

GROQ_API_KEY = get_key("GROQ_API_KEY")
GEMINI_API_KEY = get_key("GEMINI_API_KEY")

print("Groq key provided:  ", bool(GROQ_API_KEY))
print("Gemini key provided:", bool(GEMINI_API_KEY))

if not GROQ_API_KEY and not GEMINI_API_KEY:
    print("\nNo keys provided. The retrieval parts will still run, but LLM answers will be skipped.")

Enter GROQ_API_KEY (press Enter to skip): ··········
Enter GEMINI_API_KEY (press Enter to skip): ··········
Groq key provided:   True
Gemini key provided: False


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
from chromadb.utils import embedding_functions

documents = {
    "ai_overview.txt": """Artificial Intelligence: An Overview

Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.

Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own.

Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.

Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.
""",
    "embeddings_in_practice.txt": """Embeddings and Vector Search in Practice

Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
allowing the system to find results that match the meaning of the query, not just its exact words.

This is especially powerful for customer support systems, where a user might phrase a question very
differently from how it appears in a help article, yet still expect to find the right answer.

Vector databases such as ChromaDB store these embeddings and allow fast similarity search across
thousands or millions of documents at once.
"""
}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=250, chunk_overlap=40, separators=["\n\n", "\n", ". ", " ", ""]
)

client = chromadb.Client()
hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
collection = client.create_collection(name="rag_chunks", embedding_function=hf_embedder)

for source, text in documents.items():
    doc_chunks = splitter.split_text(text)
    collection.add(
        documents=doc_chunks,
        metadatas=[{"source": source, "chunk_index": i} for i in range(len(doc_chunks))],
        ids=[f"{source}_chunk_{i}" for i in range(len(doc_chunks))]
    )

print("Total chunks stored:", collection.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total chunks stored: 13


In [4]:
def retrieve_context(query, n_results=4, similarity_threshold=0.25):
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    context_chunks = []
    for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        similarity = 1 - distance
        if similarity >= similarity_threshold:
            context_chunks.append({
                "text": doc,
                "source": meta["source"],
                "chunk_index": meta["chunk_index"],
                "similarity": round(similarity, 4)
            })
    return context_chunks

# Quick check that retrieval works
test_chunks = retrieve_context("What is RAG?")
print("Chunks retrieved:", len(test_chunks))
for c in test_chunks:
    print(f"  [{c['similarity']}] {c['source']} #{c['chunk_index']}")

Chunks retrieved: 1
  [0.5667] ai_overview.txt #6


In [5]:
SYSTEM_PROMPT = (
    "You are a helpful assistant that answers questions using ONLY the provided context. "
    "If the answer is not in the context, reply exactly: \"I don't know based on the provided documents.\" "
    "Keep the answer short and clear. At the end, mention the source file name(s) you used."
)

def build_user_prompt(query, context_chunks):
    context_text = "\n\n".join(
        f"[Source: {c['source']}, chunk {c['chunk_index']}]\n{c['text']}"
        for c in context_chunks
    )
    return f"Context:\n{context_text}\n\nQuestion: {query}\n\nAnswer:"

sample_chunks = retrieve_context("How does RAG improve a language model?")
print(build_user_prompt("How does RAG improve a language model?", sample_chunks))

Context:
[Source: ai_overview.txt, chunk 6]
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG

[Source: ai_overview.txt, chunk 2]
understanding language, recognizing images, making decisions, and learning from experience.

Question: How does RAG improve a language model?

Answer:


In [6]:
from groq import Groq

GROQ_MODELS = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant", "openai/gpt-oss-20b"]

def ask_groq(user_prompt):
    if not GROQ_API_KEY:
        return "[Groq skipped: no API key provided]"

    groq_client = Groq(api_key=GROQ_API_KEY)
    last_error = None

    for model_name in GROQ_MODELS:
        try:
            response = groq_client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.1,
                max_tokens=600
            )
            answer = response.choices[0].message.content
            if answer:
                return answer.strip()
        except Exception as e:
            last_error = e

    return f"[Groq error: {last_error}]"

question = "How does RAG improve a language model?"
chunks = retrieve_context(question)
print("Groq answer:\n")
print(ask_groq(build_user_prompt(question, chunks)))

Groq answer:

[Groq error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'expired_api_key'}}]


In [7]:
from google import genai
from google.genai import types

GEMINI_MODELS = ["gemini-2.5-flash", "gemini-3-flash-preview", "gemini-2.5-flash-lite"]

def ask_gemini(user_prompt):
    if not GEMINI_API_KEY:
        return "[Gemini skipped: no API key provided]"

    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    last_error = None

    for model_name in GEMINI_MODELS:
        try:
            response = gemini_client.models.generate_content(
                model=model_name,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0.1,
                    max_output_tokens=1500
                )
            )
            if response.text:
                return response.text.strip()
        except Exception as e:
            last_error = e

    return f"[Gemini error: {last_error}]"

print("Gemini answer:\n")
print(ask_gemini(build_user_prompt(question, chunks)))

Gemini answer:

[Gemini skipped: no API key provided]


In [8]:
def rag_answer(query, provider="groq", show_sources=True):
    context_chunks = retrieve_context(query)

    if not context_chunks:
        return "I don't know based on the provided documents. (No relevant chunks were retrieved.)"

    user_prompt = build_user_prompt(query, context_chunks)

    if provider == "gemini":
        answer = ask_gemini(user_prompt)
    else:
        answer = ask_groq(user_prompt)

    if show_sources:
        sources = sorted({f"{c['source']} #{c['chunk_index']}" for c in context_chunks})
        answer += "\n\nRetrieved from: " + ", ".join(sources)

    return answer

# Pick the provider you have a key for
default_provider = "groq" if GROQ_API_KEY else "gemini"
print("Using provider:", default_provider)

Using provider: groq


In [11]:
q1 = "What is deep learning?"
print("Question:", q1, "\n")
print(rag_answer(q1, provider=default_provider))

Question: What is deep learning? 

[Groq error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'expired_api_key'}}]

Retrieved from: ai_overview.txt #1, ai_overview.txt #2, ai_overview.txt #3, ai_overview.txt #4


In [12]:
q2 = "How do embeddings and vector databases help RAG find useful information?"
print("Question:", q2, "\n")
print(rag_answer(q2, provider=default_provider))

Question: How do embeddings and vector databases help RAG find useful information? 

[Groq error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'expired_api_key'}}]

Retrieved from: ai_overview.txt #6, embeddings_in_practice.txt #0, embeddings_in_practice.txt #1, embeddings_in_practice.txt #4


In [13]:
q3 = "What is the best recipe for chocolate cake?"
print("Question:", q3, "\n")
print(rag_answer(q3, provider=default_provider))

Question: What is the best recipe for chocolate cake? 

I don't know based on the provided documents. (No relevant chunks were retrieved.)


In [10]:
compare_q = "Why is RAG more reliable than asking a language model directly?"
print("Question:", compare_q, "\n")

compare_chunks = retrieve_context(compare_q)
compare_prompt = build_user_prompt(compare_q, compare_chunks)

print("=" * 25, "GROQ", "=" * 25)
print(ask_groq(compare_prompt))

print("\n" + "=" * 24, "GEMINI", "=" * 24)
print(ask_gemini(compare_prompt))

Question: Why is RAG more reliable than asking a language model directly? 

========================= GROQ =========================
[Groq error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'expired_api_key'}}]

======================== GEMINI ========================
[Gemini skipped: no API key provided]


In [9]:
while True:
    user_question = input("Ask a question (or press Enter to quit): ").strip()
    if not user_question:
        print("Goodbye!")
        break
    print("\n" + rag_answer(user_question, provider=default_provider) + "\n")

Ask a question (or press Enter to quit): 
Goodbye!
